In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, applications

(X_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 31s 0us/step


c:\Users\konta\anaconda3\envs\tk\Lib\site-packages\keras\src\datasets\cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


In [6]:
from tensorflow.keras import layers, models, applications

def build_efficient_net_b3(input_shape=(32, 32, 3), num_classes=10):
    inputs = layers.Input(shape=input_shape)

    # Data augmentation
    x = layers.RandomFlip('horizontal')(inputs)
    x = layers.RandomRotation(0.1)(x)
    x = layers.RandomZoom(0.1)(x)

    # Resize for EfficientNet
    x = layers.Resizing(300, 300)(x)  # B3 expects ~300x300

    # Preprocessing (IMPORTANT)
    x = applications.efficientnet.preprocess_input(x)

    # Load pretrained model
    base_model = applications.EfficientNetB3(
        include_top=False,
        weights='imagenet',
        input_shape=(300, 300, 3)
    )

    base_model.trainable = False

    x = base_model(x, training=False)

    # Custom head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='swish')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)

    return model, base_model


model, base_model = build_efficient_net_b3()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_warmup = model.fit(
    X_train,
    y_train,
    epochs=5,
    validation_data=(x_test, y_test),
    batch_size=64
)

In [ ]:
base_model.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_finetune = model.fit(X_train, y_train, epochs=15, validation_data=(x_test, y_test), batch_size=12)